# laminar-coal: flat flame burner

Plots the y=0 centerline slice (`writeSurface`) and the coal cloud (`writeCloud`)
written by `flameletCloudFoam` into `laminar-coal/postProcessing/`.
Run the case first (see `../laminar-coal/README.md`), or point `CASE` at existing output.
Flow is along z.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import oxysim_viz as ov

CASE = Path(os.environ.get("OXYSIM_LAMINAR_CASE", "../laminar-coal"))
TIME = None  # None = latest available; otherwise the nearest available time is used
PLANE_TOL = 2e-3  # m -- particles within this |y| of the slice are drawn

ov.set_style()

In [ ]:
mesh, t = ov.load_slice(CASE, "writeSurface", "slice.vtp", TIME)
cloud, t_cloud = ov.load_cloud(CASE, "writeCloud", "coalCloud", t)
print(f"slice t = {t}, cloud t = {t_cloud}, {cloud.n_points} particles")
print("slice fields:", list(mesh.point_data.keys()))

## Temperature with coal particles

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=200)
ov.plot_slice(ax, mesh, "T", cloud=cloud, cloud_color="T", plane_tol=PLANE_TOL,
              cmap="inferno", particle_size=6)
ax.set_title(f"t = {t} s")
plt.show()

## Gas-phase fields

In [ ]:
fields = {"T": "inferno", "Z": "viridis", "yc": "cividis", "|U|": "magma", "OH": "plasma"}
fig, axs = plt.subplots(1, len(fields), figsize=(4.5 * len(fields), 4), dpi=200, gridspec_kw={"wspace": 0.8})
for ax, (f, cmap) in zip(axs, fields.items()):
    ov.plot_slice(ax, mesh, f, cmap=cmap)
plt.show()

## Maxima over time

Shows how the flame cone establishes and then plateaus. Reads every `STRIDE`-th slice.

In [ ]:
STRIDE = 10
times = [tn[0] for tn in ov.sample_times(CASE / "postProcessing" / "writeSurface")][::STRIDE]
maxima = {f: [] for f in ("T", "Z", "yc")}
for tt in times:
    m, _ = ov.load_slice(CASE, "writeSurface", "slice.vtp", tt)
    for f in maxima:
        maxima[f].append(ov.values(m, f).max())

fig, axs = plt.subplots(1, 3, figsize=(12, 3), dpi=200)
for ax, (f, v) in zip(axs, maxima.items()):
    ax.plot(times, v)
    ax.set_xlabel(ov.LABELS["t"])
    ax.set_ylabel(f"max {ov.LABELS[f]}")
    ax.grid(alpha=0.5)
fig.tight_layout()
plt.show()